# ARDY on Kaggle — T4 × 2 FP16 Sharded

这版用于 **2× NVIDIA T4（每张约 14.5 GiB）**：

- LLM2Vec / Llama-3 8B：**原生 FP16，不量化**
- 一个 Llama 跨两张 T4 做 **model sharding**，不是 DataParallel
- `balanced_low_0` 尽量把更多 Llama 权重放在 GPU1，给 GPU0 的 ARDY 留空间
- GPU0 Llama 上限默认 `9.5GiB`，GPU1 默认 `12GiB`
- ARDY Core8 固定在 `cuda:0`
- 文本 embedding 输出回到 `cuda:0` 给 ARDY 使用
- 修复 LLM2Vec 双 GPU multiprocessing / `.to(device)` 会破坏分片的问题
- 保留 Prompt embedding cache、自定义 3D 播放器、Cloudflare Tunnel
- ARDY commit：`693f74d13b3d04a0a22ce127ee79c929dd89756b`

第一次运行 Cell 0 会安装依赖并自动重启 Kernel 一次；重启后从 Cell 0 重新运行。


In [ ]:
# Cell 0 — Bootstrap + FP16 sharding patch
from pathlib import Path
import os, signal, subprocess, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = Path("/kaggle/working")
REPO = ROOT / "ardy"
ARDY_COMMIT = "693f74d13b3d04a0a22ce127ee79c929dd89756b"
MARKER = ROOT / ".ardy_t4x2_fp16_sharded_v1_ready"

def run(cmd, cwd=None):
    cmd = list(map(str, cmd))
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO.exists():
    run(["git", "clone", "https://github.com/nv-tlabs/ardy.git", REPO])
run(["git", "fetch", "origin", ARDY_COMMIT, "--depth", "1"], cwd=REPO)
run(["git", "checkout", "--detach", ARDY_COMMIT], cwd=REPO)
run(["git", "reset", "--hard", ARDY_COMMIT], cwd=REPO)

# 修复 ardy/assets.py 与 ardy/assets/ 的命名冲突（在 import ardy 前完成）
assets_init = REPO / "ardy/assets/__init__.py"
assets_init.write_text('''from pathlib import Path\nASSETS_ROOT=Path(__file__).parent\nDEMO_ASSETS_ROOT=ASSETS_ROOT/"demo"\nDEMO_EXAMPLES_ROOT=DEMO_ASSETS_ROOT/"examples"\nSKELETONS_ROOT=ASSETS_ROOT/"skeletons"\nSOMA_ASSETS_ROOT=ASSETS_ROOT/"SOMA"\ndef skeleton_asset_path(*parts): return SKELETONS_ROOT.joinpath(*parts)\ndef demo_asset_path(*parts): return DEMO_ASSETS_ROOT.joinpath(*parts)\n''')

# LLM2Vec 双 GPU 原版会 multiprocessing，并在 _encode() 里 self.to(device)。
# Accelerate 分片模型不能整体搬卡，因此显式 device 时走单进程，并用 _ardy_sharded 禁止整体 .to()。
llm2vec_py = REPO / "ardy/model/llm2vec/llm2vec.py"
src = llm2vec_py.read_text()
old = '        if torch.cuda.device_count() <= 1:\n            # This branch also support mps devices\n            self.to(device)'
new = '        if torch.cuda.device_count() <= 1 or device is not None:\n            # Kaggle T4x2 FP16 sharding: explicit device means single-process encode.\n            if not getattr(self, "_ardy_sharded", False):\n                self.to(device)'
if old not in src:
    raise RuntimeError("LLM2Vec encode patch target changed")
src = src.replace(old, new, 1)
old = '        self.to(device)\n        features = self.tokenize([self.prepare_for_tokenization(sentence) for sentence in sentences_batch])'
new = '        if not getattr(self, "_ardy_sharded", False):\n            self.to(device)\n        features = self.tokenize([self.prepare_for_tokenization(sentence) for sentence in sentences_batch])'
if old not in src:
    raise RuntimeError("LLM2Vec _encode patch target changed")
src = src.replace(old, new, 1)
llm2vec_py.write_text(src)

# 专用 FP16 双 T4 encoder：Llama 跨卡；token 输入送 embedding 所在卡；最终 4096-d embedding 回 cuda:0。
shard_module = REPO / "ardy/model/kaggle_t4_fp16_text_encoder.py"
shard_module.write_text(r'''import os
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer
from .llm2vec.llm2vec import LLM2Vec
from .llm2vec.models.bidirectional_llama import LlamaBiModel

LLAMA_REPO = "meta-llama/Meta-Llama-3-8B-Instruct"
MNTP_REPO = "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp"
SUPERVISED_REPO = "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised"

class ShardedFP16LLM2VecEncoder:
    def __init__(self, output_device="cuda:0", max_memory=None, llm_dim=4096):
        if torch.cuda.device_count() < 2:
            raise RuntimeError("需要 2 张 CUDA GPU")
        self.llm_dim = llm_dim
        self._output_device = str(output_device)
        self.device = torch.device(self._output_device)
        self.dtype = torch.float16
        self.max_memory = max_memory or {0: "9.5GiB", 1: "12GiB"}
        cache_dir = os.environ.get("HUGGINGFACE_CACHE_DIR")
        token = os.environ.get("HF_TOKEN")

        print("Loading Llama-3 8B base as bidirectional FP16 shards...", flush=True)
        print("device_map=balanced_low_0 | max_memory=", self.max_memory, flush=True)
        tokenizer = AutoTokenizer.from_pretrained(MNTP_REPO, cache_dir=cache_dir, token=token)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"

        # 直接加载真正的 Llama base，避免把 adapter repo 当完整 base 权重目录。
        base = LlamaBiModel.from_pretrained(
            LLAMA_REPO,
            torch_dtype=torch.float16,
            device_map="balanced_low_0",
            max_memory=self.max_memory,
            low_cpu_mem_usage=True,
            cache_dir=cache_dir,
            token=token,
        )

        # 与 ARDY/LLM2Vec 原逻辑保持一致：先合并 MNTP LoRA，再挂 supervised LoRA。
        print("Applying MNTP adapter...", flush=True)
        base = PeftModel.from_pretrained(base, MNTP_REPO, low_cpu_mem_usage=True)
        base = base.merge_and_unload()
        print("Applying supervised adapter...", flush=True)
        model = PeftModel.from_pretrained(base, SUPERVISED_REPO, low_cpu_mem_usage=True)

        self.model = LLM2Vec(model=model, tokenizer=tokenizer)
        self.model._ardy_sharded = True
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

        emb = self.model.model.get_input_embeddings()
        self._input_device = str(emb.weight.device)
        if not self._input_device.startswith("cuda:"):
            raise RuntimeError(f"Embedding layer landed on {self._input_device}; GPU-only sharding expected")

        by_dev = {}
        for p in self.model.parameters():
            key = str(p.device)
            by_dev[key] = by_dev.get(key, 0) + p.numel() * p.element_size()
        self.parameter_bytes_by_device = by_dev
        cuda_used = {k for k, v in by_dev.items() if k.startswith("cuda:") and v > 0}
        if not {"cuda:0", "cuda:1"}.issubset(cuda_used):
            raise RuntimeError(f"模型没有真正拆到两张 T4: {by_dev}")
        cpu_gib = by_dev.get("cpu", 0) / 1024**3
        if cpu_gib > 0.2:
            raise RuntimeError(f"出现 {cpu_gib:.2f} GiB CPU offload；请降低模型显存占用或提高 GPU max_memory")

        print("Input device:", self._input_device)
        for dev, n in sorted(by_dev.items()):
            print(f"Parameters on {dev}: {n/1024**3:.2f} GiB")

    def to(self, device=None, dtype=None):
        # 只改变 embedding 输出位置/精度；绝不整体移动已经分片的 8B 模型。
        if device is not None:
            self._output_device = str(device)
            self.device = torch.device(device)
        if dtype is not None:
            self.dtype = dtype
        return self

    def eval(self):
        self.model.eval()
        return self

    def get_device(self):
        return self._output_device

    def __call__(self, text):
        is_string = isinstance(text, str)
        texts = [text] if is_string else list(text)
        with torch.inference_mode():
            encoded = self.model.encode(
                texts,
                batch_size=1,
                show_progress_bar=False,
                convert_to_tensor=True,
                device=self._input_device,
            )
        encoded = encoded[:, None].to(device=self._output_device, dtype=self.dtype)
        lengths = np.ones(len(texts), dtype=int).tolist()
        return (encoded[0], lengths[0]) if is_string else (encoded, lengths)

def build_fp16_text_encoder(output_device="cuda:0", max_memory=None):
    return ShardedFP16LLM2VecEncoder(output_device=output_device, max_memory=max_memory)
''')

if not MARKER.exists():
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", ".[demo]", "accelerate==1.14.0"], cwd=REPO)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
    MARKER.write_text(ARDY_COMMIT + "\n")
    print("Installed; restarting Kaggle kernel once.")
    os.kill(os.getpid(), signal.SIGKILL)

print("Bootstrap ready:", REPO)


In [ ]:
# Cell 1 — Runtime / HF Token / 双 T4 验证
from pathlib import Path
import os, sys, importlib.metadata as metadata

ROOT = Path("/kaggle/working")
REPO = ROOT / "ardy"
WORK_CACHE = ROOT / "ardy-cache-fp16"
HF_CACHE = WORK_CACHE / "huggingface"
CHECKPOINTS_DIR = WORK_CACHE / "checkpoints"
EMBED_CACHE = WORK_CACHE / "text_embedding_cache"
for p in (HF_CACHE, CHECKPOINTS_DIR, EMBED_CACHE): p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HUGGINGFACE_CACHE_DIR"] = str(HF_CACHE)
os.environ["CHECKPOINTS_DIR"] = str(CHECKPOINTS_DIR)
os.environ["LOCAL_CACHE"] = "true"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "YES"
os.environ["ARDY_TEXT_PRECISION"] = "fp16-sharded"

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
assert HF_TOKEN, "请在 Kaggle Secrets 添加 HF_TOKEN，并确保账号已获 Meta-Llama-3-8B-Instruct 权限"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("HF_TOKEN loaded (not printed).")

import numpy as np, torch
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)
print("Transformers", metadata.version("transformers"), "| Accelerate", metadata.version("accelerate"))
assert np.__version__.startswith("1.26."), f"需要 NumPy 1.26.x，当前 {np.__version__}"
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "请在 Kaggle 选择 GPU T4 x2"
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {p.name} | {p.total_memory/1024**3:.2f} GiB | CC {p.major}.{p.minor}")

from ardy.model.kaggle_t4_fp16_text_encoder import build_fp16_text_encoder
from ardy.model import load_model
print("FP16 sharding patch imports OK")


In [ ]:
# Cell 2 — Hugging Face 登录 + gated model 权限验证
from huggingface_hub import login, hf_hub_download
login(token=HF_TOKEN, add_to_git_credential=False)
hf_hub_download("meta-llama/Meta-Llama-3-8B-Instruct", "config.json", cache_dir=str(HF_CACHE), token=HF_TOKEN)
print("Meta-Llama-3-8B-Instruct access OK")


In [ ]:
# Cell 3 — 下载 ARDY + LLM2Vec/Llama 权重到 HF cache
from huggingface_hub import snapshot_download

ARDY_MODEL_NAME = "ARDY-Core-RP-20FPS-Horizon8"
ARDY_LOCAL = CHECKPOINTS_DIR / ARDY_MODEL_NAME

if not (ARDY_LOCAL / "config.yaml").exists():
    snapshot_download("nvidia/" + ARDY_MODEL_NAME, local_dir=str(ARDY_LOCAL), token=HF_TOKEN)
else:
    print("ARDY ready:", ARDY_LOCAL)

for repo_id in [
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised",
]:
    print("Caching:", repo_id)
    snapshot_download(repo_id, cache_dir=str(HF_CACHE), token=HF_TOKEN)
print("All model files ready.")


In [ ]:
# Cell 4 — LLM2Vec / Llama-3 8B FP16 跨两张 T4 分片
import gc, sys, time, torch

# 若重复执行，先释放旧对象；第一次执行无影响。
if "web_server" in globals():
    try:
        web_server.shutdown(); web_server.server_close()
    except Exception:
        pass
for name in ["model", "text_encoder", "raw_text_encoder", "motion", "output"]:
    globals().pop(name, None)
gc.collect()
for i in (0, 1):
    with torch.cuda.device(i):
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(i)

for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"Before LLM | GPU{i} free={free/1024**3:.2f}/{total/1024**3:.2f} GiB")
assert torch.cuda.mem_get_info(0)[0] > 13*1024**3 and torch.cuda.mem_get_info(1)[0] > 13*1024**3, "GPU 不干净；建议 Restart Kernel 后从 Cell 0 顺序运行"

from ardy.model.kaggle_t4_fp16_text_encoder import build_fp16_text_encoder

# GPU0 要给 ARDY 留空间；GPU1 给 activation / CUDA runtime 留约 2.5GiB。
LLM_MAX_MEMORY = {0: "9.5GiB", 1: "12GiB"}
t0 = time.perf_counter()
raw_text_encoder = build_fp16_text_encoder(output_device="cuda:0", max_memory=LLM_MAX_MEMORY)
for i in (0, 1): torch.cuda.synchronize(i)
print(f"FP16 sharded load: {time.perf_counter()-t0:.2f}s")

scripts_dir = str(REPO / "scripts")
if scripts_dir not in sys.path: sys.path.insert(0, scripts_dir)
from interactive_demo.embedding_cache import CachedTextEncoder
text_encoder = CachedTextEncoder(raw_text_encoder, cache_dir=str(EMBED_CACHE))

t0 = time.perf_counter()
emb, length = text_encoder("A person walks forward.")
for i in (0, 1): torch.cuda.synchronize(i)
print("\n===== FP16 SHARDED LLM2VEC OK =====")
print("Embedding:", tuple(emb.shape), emb.dtype, emb.device, "| encode", f"{time.perf_counter()-t0:.3f}s")
print("LLM input device:", raw_text_encoder._input_device)
for dev, n in sorted(raw_text_encoder.parameter_bytes_by_device.items()):
    print(f"LLM parameters {dev}: {n/1024**3:.2f} GiB")
for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} | peak={torch.cuda.max_memory_allocated(i)/1024**3:.2f} | free={free/1024**3:.2f} GiB")


In [ ]:
# Cell 5 — ARDY Core8 -> cuda:0（与 Llama shard 共存）
import gc, torch
MOTION_DEVICE = "cuda:0"
gc.collect()
with torch.cuda.device(0):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
free0, total0 = torch.cuda.mem_get_info(0)
print(f"Before ARDY | GPU0 free={free0/1024**3:.2f} GiB")
assert free0 > 3.0*1024**3, "GPU0 给 ARDY 的余量不足；把 LLM_MAX_MEMORY[0] 再调低"

from ardy.model import load_model
model = load_model("core8", device=MOTION_DEVICE, text_encoder=text_encoder, checkpoints_dir=str(CHECKPOINTS_DIR))
model.eval()
print("\n===== ARDY OK =====")
print(f"FPS={model.motion_rep.fps} | Horizon={model.gen_horizon_len} | Frames/token={model.num_frames_per_token}")
for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} | reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} | free={free/1024**3:.2f} GiB")


In [ ]:
# Cell 6 — Smoke Test
import time, torch
from ardy.motion_rep.tools import length_to_mask

fps = float(model.motion_rep.fps)
num_base_steps = int(model.diffusion.num_base_steps)
prompt = "A person walks forward."
duration = 2.0
num_frames = int(duration * fps)
pad_mask = length_to_mask(torch.tensor([num_frames], device=MOTION_DEVICE))
first_heading_angle = torch.zeros(1, device=MOTION_DEVICE)

for i in (0, 1): torch.cuda.synchronize(i)
t0 = time.perf_counter()
with torch.inference_mode():
    motion = model(
        [prompt], num_frames,
        num_denoising_steps=num_base_steps,
        pad_mask=pad_mask,
        first_heading_angle=first_heading_angle,
        motion_mask=None,
        observed_motion=None,
        cfg_weight=(2.0, 2.0),
        crop_history_length=model.num_frames_per_token,
    )
    output = model.motion_rep.inverse(motion, is_normalized=True)
for i in (0, 1): torch.cuda.synchronize(i)
dt = time.perf_counter() - t0
print(f"Prompt: {prompt}\nFrames: {num_frames}\nGeneration: {dt:.3f}s\nRTF: {dt/duration:.3f}\nShape: {' × '.join(map(str, output['posed_joints'].shape))}")
for i in (0, 1):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} | peak={torch.cuda.max_memory_allocated(i)/1024**3:.2f} | free={free/1024**3:.2f} GiB")


In [ ]:
# Cell 7 — 保存 Smoke Test NPZ
from pathlib import Path
import numpy as np
OUT = Path("/kaggle/working/ardy_outputs")
OUT.mkdir(parents=True, exist_ok=True)
posed_joints = output["posed_joints"].detach().float().cpu().numpy()
np.savez(OUT / "smoke_test_fp16.npz", posed_joints=posed_joints, fps=fps, prompt=prompt)
print("Saved:", OUT / "smoke_test_fp16.npz")


## Interactive 3D Player

这里直接复用当前 Kernel 里已经加载好的 **FP16 双卡 LLM2Vec + ARDY**，不会释放模型，也不会再启动官方 Demo 子进程。


In [ ]:
# Cell 8a — 关闭旧 Web Server（重复运行播放器时使用）
if "web_server" in globals():
    try:
        web_server.shutdown(); web_server.server_close(); print("Old web server closed.")
    except Exception as e:
        print("Close old server:", e)
if "web_thread" in globals():
    try: web_thread.join(timeout=2)
    except: pass


In [ ]:
# Cell 8 — ARDY 自定义 3D 动作播放器（FP16 双卡）
import json, time, threading, urllib.parse, uuid
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from pathlib import Path

import numpy as np
import torch
from ardy.motion_rep.tools import length_to_mask

assert "model" in globals(), "请先运行 Cell 4 加载 ARDY"
assert "text_encoder" in globals(), "请先运行 Cell 3 加载 LLM2Vec"

HOST, PORT = "0.0.0.0", 2333
OUT_DIR = Path("/kaggle/working/ardy_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
gpu_lock = threading.Lock()

HTML = r"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ARDY T4×2</title>

<style>
body{margin:0;background:#111;color:#eee;font-family:Arial,sans-serif}
main{max-width:1050px;margin:auto;padding:20px}
textarea,input,button{box-sizing:border-box;padding:10px;font-size:15px;border-radius:7px}
textarea,input{background:#222;color:white;border:1px solid #555}
textarea{width:100%;height:70px}
button{cursor:pointer;border:0;padding:10px 18px;font-weight:bold}
.row{display:flex;gap:10px;align-items:center;margin:10px 0}
#viewer{width:100%;height:600px;background:#080808;border-radius:10px;overflow:hidden}
#timeline{width:100%}
pre{background:#222;padding:12px;border-radius:8px;white-space:pre-wrap}
a{color:#7dc8ff}
</style>

<script type="importmap">
{
  "imports":{
    "three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
    "three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"
  }
}
</script>
</head>

<body>
<main>

<h2>ARDY — T4 × 2</h2>

<label>Prompt</label>
<textarea id="prompt">A person walks forward.</textarea>

<div class="row">
<label>Duration</label>
<input id="duration" type="number" value="2" min="0.5" max="10" step="0.5">
<span>seconds</span>
<button id="generate">Generate</button>
</div>

<div id="viewer"></div>

<div class="row">
<button id="play">▶ Play</button>
<span id="frameLabel">Frame 0 / 0</span>
</div>

<input id="timeline" type="range" min="0" max="0" value="0">

<pre id="result">Ready.</pre>

<script type="module">

import * as THREE from "three";
import {OrbitControls} from "three/addons/controls/OrbitControls.js";

const viewer=document.getElementById("viewer");

const scene=new THREE.Scene();
scene.background=new THREE.Color(0x080808);

const camera=new THREE.PerspectiveCamera(45,viewer.clientWidth/viewer.clientHeight,0.01,1000);
camera.position.set(3,2,5);

const renderer=new THREE.WebGLRenderer({antialias:true});
renderer.setPixelRatio(Math.min(devicePixelRatio,2));
renderer.setSize(viewer.clientWidth,viewer.clientHeight);
viewer.appendChild(renderer.domElement);

const controls=new OrbitControls(camera,renderer.domElement);
controls.enableDamping=true;

scene.add(new THREE.HemisphereLight(0xffffff,0x333333,3));

const grid=new THREE.GridHelper(10,20);
scene.add(grid);

const axes=new THREE.AxesHelper(1);
scene.add(axes);

let frames=[];
let fps=20;
let current=0;
let playing=false;
let lastTime=performance.now();

let joints=[];
let bones=null;
let edges=[];


/* 根据第一帧做最小生成树，自动得到大致骨架连线 */
function buildMST(points){

    const n=points.length;
    if(n<2)return [];

    const used=new Array(n).fill(false);
    const dist=new Array(n).fill(Infinity);
    const parent=new Array(n).fill(-1);

    dist[0]=0;

    for(let step=0;step<n;step++){

        let u=-1,best=Infinity;

        for(let i=0;i<n;i++){
            if(!used[i] && dist[i]<best){
                best=dist[i];
                u=i;
            }
        }

        if(u<0)break;
        used[u]=true;

        for(let v=0;v<n;v++){

            if(used[v])continue;

            const dx=points[u][0]-points[v][0];
            const dy=points[u][1]-points[v][1];
            const dz=points[u][2]-points[v][2];

            const d=dx*dx+dy*dy+dz*dz;

            if(d<dist[v]){
                dist[v]=d;
                parent[v]=u;
            }
        }
    }

    const result=[];

    for(let i=1;i<n;i++){
        if(parent[i]>=0)result.push([parent[i],i]);
    }

    return result;
}


function createSkeleton(count){

    for(const j of joints)scene.remove(j);
    joints=[];

    if(bones){
        scene.remove(bones);
        bones.geometry.dispose();
    }

    const geo=new THREE.SphereGeometry(0.035,12,12);
    const mat=new THREE.MeshStandardMaterial({color:0x55bbff});

    for(let i=0;i<count;i++){
        const mesh=new THREE.Mesh(geo,mat);
        scene.add(mesh);
        joints.push(mesh);
    }

    const linePositions=new Float32Array(edges.length*6);
    const lineGeo=new THREE.BufferGeometry();
    lineGeo.setAttribute("position",new THREE.BufferAttribute(linePositions,3));

    bones=new THREE.LineSegments(
        lineGeo,
        new THREE.LineBasicMaterial({color:0xffffff})
    );

    scene.add(bones);
}


function updateFrame(index){

    if(!frames.length)return;

    current=Math.max(0,Math.min(index,frames.length-1));

    const frame=frames[current];

    for(let i=0;i<frame.length;i++){
        const p=frame[i];

        /* ARDY XYZ -> Three XYZ */
        joints[i].position.set(p[0],p[1],p[2]);
    }

    const pos=bones.geometry.attributes.position.array;

    let k=0;

    for(const [a,b] of edges){

        const pa=frame[a];
        const pb=frame[b];

        pos[k++]=pa[0];
        pos[k++]=pa[1];
        pos[k++]=pa[2];

        pos[k++]=pb[0];
        pos[k++]=pb[1];
        pos[k++]=pb[2];
    }

    bones.geometry.attributes.position.needsUpdate=true;

    document.getElementById("timeline").value=current;
    document.getElementById("frameLabel").textContent=`Frame ${current+1} / ${frames.length}`;
}


function fitCamera(){

    if(!frames.length)return;

    const box=new THREE.Box3();

    for(const frame of frames){
        for(const p of frame){
            box.expandByPoint(new THREE.Vector3(p[0],p[1],p[2]));
        }
    }

    const center=new THREE.Vector3();
    const size=new THREE.Vector3();

    box.getCenter(center);
    box.getSize(size);

    const span=Math.max(size.x,size.y,size.z,1);

    controls.target.copy(center);

    camera.position.set(
        center.x+span*1.7,
        center.y+span*1.1,
        center.z+span*1.7
    );

    camera.near=span/100;
    camera.far=span*100;
    camera.updateProjectionMatrix();

    controls.update();
}


function animate(now){

    requestAnimationFrame(animate);

    if(playing && frames.length){

        const frameTime=1000/fps;

        if(now-lastTime>=frameTime){

            lastTime=now;

            current++;

            if(current>=frames.length)current=0;

            updateFrame(current);
        }
    }

    controls.update();
    renderer.render(scene,camera);
}

requestAnimationFrame(animate);


document.getElementById("play").onclick=()=>{

    playing=!playing;

    document.getElementById("play").textContent=
        playing ? "⏸ Pause" : "▶ Play";

    lastTime=performance.now();
};


document.getElementById("timeline").oninput=e=>{

    playing=false;
    document.getElementById("play").textContent="▶ Play";

    updateFrame(parseInt(e.target.value));
};


document.getElementById("generate").onclick=async()=>{

    const result=document.getElementById("result");

    result.textContent="Generating...";

    try{

        const response=await fetch("/generate",{
            method:"POST",
            headers:{"Content-Type":"application/json"},
            body:JSON.stringify({
                prompt:document.getElementById("prompt").value,
                duration:parseFloat(document.getElementById("duration").value)
            })
        });

        const data=await response.json();

        if(!response.ok)throw new Error(data.error || "Generate failed");

        frames=data.joints;
        fps=data.fps;
        current=0;

        edges=buildMST(frames[0]);

        createSkeleton(frames[0].length);

        const timeline=document.getElementById("timeline");
        timeline.max=frames.length-1;
        timeline.value=0;

        updateFrame(0);
        fitCamera();

        playing=true;
        document.getElementById("play").textContent="⏸ Pause";
        lastTime=performance.now();

        result.innerHTML=
            `Prompt: ${data.prompt}\n`+
            `Frames: ${data.frames}\n`+
            `FPS: ${data.fps}\n`+
            `Generation: ${data.seconds.toFixed(3)} s\n`+
            `RTF: ${data.rtf.toFixed(3)}\n`+
            `Shape: ${data.shape.join(" × ")}\n\n`+
            `<a href="${data.download}">Download NPZ</a>`;

    }catch(e){

        result.textContent="ERROR: "+e;
    }
};


window.addEventListener("resize",()=>{

    camera.aspect=viewer.clientWidth/viewer.clientHeight;
    camera.updateProjectionMatrix();

    renderer.setSize(viewer.clientWidth,viewer.clientHeight);
});

</script>
</main>
</body>
</html>
"""


class Handler(BaseHTTPRequestHandler):

    def log_message(self,fmt,*args):
        print("[WEB]",fmt%args)

    def send_data(self,data,ctype="text/html; charset=utf-8",status=200):
        self.send_response(status)
        self.send_header("Content-Type",ctype)
        self.send_header("Content-Length",str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):

        if self.path=="/":
            return self.send_data(HTML.encode())

        if self.path=="/health":
            return self.send_data(b'{"status":"ok"}',"application/json")

        if self.path.startswith("/download/"):

            name=Path(urllib.parse.unquote(self.path.split("/download/",1)[1])).name
            file=OUT_DIR/name

            if not file.exists():
                return self.send_data(b"Not found","text/plain",404)

            return self.send_data(file.read_bytes(),"application/octet-stream")

        return self.send_data(b"Not found","text/plain",404)


    def do_POST(self):

        if self.path!="/generate":
            return self.send_data(b'{"error":"not found"}',"application/json",404)

        try:

            n=int(self.headers.get("Content-Length","0"))
            req=json.loads(self.rfile.read(n))

            prompt=str(req.get("prompt","")).strip()
            duration=float(req.get("duration",2.0))

            if not prompt:
                raise ValueError("Prompt cannot be empty")

            duration=max(0.5,min(duration,10.0))

            fps=float(model.motion_rep.fps)
            num_frames=max(1,int(duration*fps))

            pad_mask=length_to_mask(torch.tensor([num_frames],device="cuda:0"))
            heading=torch.zeros(1,device="cuda:0")

            with gpu_lock:

                torch.cuda.synchronize(0)
                t0=time.perf_counter()

                with torch.inference_mode():

                    motion=model(
                        [prompt],
                        num_frames,
                        num_denoising_steps=int(model.diffusion.num_base_steps),
                        pad_mask=pad_mask,
                        first_heading_angle=heading,
                        motion_mask=None,
                        observed_motion=None,
                        cfg_weight=(2.0,2.0),
                        crop_history_length=model.num_frames_per_token,
                    )

                    output=model.motion_rep.inverse(
                        motion,
                        is_normalized=True
                    )

                torch.cuda.synchronize(0)
                dt=time.perf_counter()-t0

            joints=output["posed_joints"].detach().float().cpu().numpy()

            filename=f"ardy_{uuid.uuid4().hex[:10]}.npz"

            np.savez(
                OUT_DIR/filename,
                posed_joints=joints,
                fps=fps,
                prompt=prompt
            )

            response={
                "prompt":prompt,
                "frames":num_frames,
                "fps":fps,
                "seconds":dt,
                "rtf":dt/duration,
                "shape":list(joints.shape),
                "joints":joints[0].tolist(),
                "download":f"/download/{filename}"
            }

            return self.send_data(
                json.dumps(response).encode(),
                "application/json"
            )

        except Exception as e:

            print("Generate error:",repr(e))

            return self.send_data(
                json.dumps({"error":str(e)}).encode(),
                "application/json",
                500
            )


if "web_server" in globals():
    try: web_server.shutdown()
    except: pass

web_server=ThreadingHTTPServer((HOST,PORT),Handler)
web_thread=threading.Thread(target=web_server.serve_forever,daemon=True)
web_thread.start()

print("ARDY 3D Player ready")
print("http://127.0.0.1:2333")

In [ ]:
# Cell 9 — Cloudflare Tunnel（先确认本地播放器健康）
from pathlib import Path
import subprocess, urllib.request, re, time

for _ in range(30):
    try:
        if urllib.request.urlopen("http://127.0.0.1:2333/health", timeout=1).status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("本地 2333 服务未就绪，请先运行 Cell 8")

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    try: tunnel_proc.wait(timeout=5)
    except: tunnel_proc.kill()

CF = Path("/kaggle/working/cloudflared")
if not CF.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)

tunnel_proc = subprocess.Popen(
    [str(CF), "tunnel", "--url", "http://127.0.0.1:2333", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line: continue
    print(line.rstrip())
    m = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
assert url, "Cloudflare Tunnel 未返回 URL"
print("\n打开：", url)


## 可选：构建 patched ARDY wheel

这个 wheel 包含本 notebook 对 LLM2Vec sharding 与 `ardy.assets` 的补丁。ARDY 的 native extension 是 CMake/C++ 扩展，并不是 CUDA/T4 专属扩展。


In [ ]:
# Cell 10 — 可选构建 patched ARDY wheel
from pathlib import Path
import hashlib, subprocess, sys
WHEEL_OUT = Path("/kaggle/working/ardy_wheels_fp16")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-w", str(WHEEL_OUT)], cwd=str(REPO), check=True)
wheel = max(WHEEL_OUT.glob("ardy-*.whl"), key=lambda p: p.stat().st_mtime)
print("Wheel:", wheel)
print("SHA256:", hashlib.sha256(wheel.read_bytes()).hexdigest())


## 执行顺序

首次：运行 **Cell 0** → 自动重启一次。

重启后依次：**0 → 1 → 2 → 3 → 4 → 5 → 6 → 7**。

网页 3D 播放：**8a → 8 → 9**。

如果 Cell 4 成功，应看到 `FP16 SHARDED LLM2VEC OK`，并且 `LLM parameters cuda:0`、`cuda:1` 都大于 0；这才说明一个 Llama 真正拆到了两张 T4。
